<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Courtside Dynamics: SB3 training

One notebook for the whole curriculum. Pick an environment (`BallBalance`, `BallBounce`, `WallBall`) and an algorithm (`SAC` or `PPO`) at the top, then run all cells.

Each environment's defaults (training budget, custom CSV rows, phase labels for the state-machine reward) live in `courtside_dynamics.recipes`, so adding an env is one entry in that registry -- this notebook needs no edits.

## 1. Install

In [ ]:
# Git branch, tag, or commit SHA to install. Keep `main` for normal runs;
# set this to an unmerged PR branch when smoke-testing its environment.
REPO_REF = "main"

!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics@{REPO_REF}"
print(f"Installed courtside-dynamics from ref: {REPO_REF}")


## 2. Choose environment & algorithm

Set `ENV` and `ALGO` here -- everything below picks them up automatically.

* `USE_DRIVE = True` mounts Google Drive so checkpoints, eval npz, replay videos, and learning-curve plots survive a Colab runtime restart. Falls back to local logs if Drive isn't available.
* `QUICK_TEST = True` runs the whole pipeline (training, evaluation, video, plot) in a couple of minutes against a tiny budget. Use it to smoke-test a new runtime before committing to a real run.

### Suggested `n_envs` on Colab L4 (24 GB VRAM) + High-RAM (~51 GB)

These MuJoCo envs are CPU-cheap (tens of microseconds per step) and the MLP policy is small, so the GPU is never the bottleneck. The numbers below assume the default `DummyVecEnv` and the ~8 vCPUs Colab gives you.

| Algo | Suggested `n_envs` | Why |
|------|--------------------|-----|
| SAC  | **8**              | Off-policy. More envs fill the replay buffer faster, and the training helper sets `gradient_steps=-1` so updates scale 1:1 with steps collected regardless of `n_envs`. Past ~8 envs the CPU rollout cost dominates. |
| PPO  | **16**             | On-policy. The rollout buffer is `n_steps * n_envs`, so throughput scales close to linearly with `n_envs`. 16 fits Colab's CPU/RAM budget; bump to 32 if you switch to `SubprocVecEnv`. |

Leaving `N_ENVS = None` uses a recipe-specific worker count when provided, then falls back to these suggestions; set an int to override.

In [ ]:
ENV = "WallBall"   # BallBalance | BallBounce | WallBall | HumanoidTennisStage0Intercept | HumanoidTennisStage1AnchoredReturn | HumanoidTennisStage2RandomizedReturn | HumanoidTennisCoopSmoke
ALGO = None           # None uses the recipe default; or set SAC | PPO
USE_DRIVE = True
QUICK_TEST = False

# Master seed forwarded to SB3 and all helper envs. Seeded by default so
# every run is reproducible end-to-end (helper envs get derived,
# non-overlapping seeds); set None for a nondeterministic run.
SEED = 0

# Training budget CEILING. With EARLY_STOP_PATIENCE set below, the run
# stops as soon as eval reward plateaus, so a generous ceiling costs
# nothing -- the first WallBall run peaked at 1.2M steps and spent the
# remaining 10+ GPU-hours collapsing past it. None falls back to the
# recipe default. An explicit value here also overrides the QUICK_TEST
# budget.
TOTAL_TIMESTEPS = None

# Parallel training workers. None preserves a recipe-specific value,
# then falls back to the table above; set an int to override.
N_ENVS = None

# Stop training after this many consecutive evaluations without a new
# best mean reward (the same count is used as warm-up, so at least 2x
# this many evaluations happen before a stop can fire). At the default
# eval_freq=25k, 20 evals = 500k env steps of no improvement. Set None
# to always train the full budget.
EARLY_STOP_PATIENCE = 20

# Extra kwargs for the SB3 algorithm constructor. For SAC on WallBall
# the entropy temperature is pinned and the replay buffer enlarged.
# The run measurements below came from the legacy 5-action environment;
# WallBall 0.8 uses 3 target actions and requires a fresh run (no old
# model, VecNormalize statistics, or replay buffer):
# auto-tuned alpha collapsed in both real runs to date (to ~0.004 in
# the 20260710 run and to ~0.0005 within the first 150k steps of the
# 20260712 run, which then plateaued at ~2-3 exchanges per rally), and
# the default 1M buffer had evicted everything before ~1.5M steps by
# the time the 20260712 run stopped at 2.55M. Set back to {} for a
# clean-defaults baseline.
# gamma 0.995 doubles the critic's effective horizon to ~200 steps
# (~1.6 rally exchanges at ~127 steps each; the 0.99 default sees less
# than ONE exchange ahead), so keeping the rally returnable is worth
# something at decision time, not just reaching the wall.
MODEL_KWARGS = {"ent_coef": 0.02, "buffer_size": 2_000_000, "gamma": 0.995}


## 3. Mount Google Drive (optional) and pick a run directory

Each call to `resolve_run_dir(ENV, ALGO)` creates a fresh, timestamped directory so re-runs don't clobber prior artifacts. Layout:

```
<root>/<env>/<algo>/<YYYYMMDD_HHMMSS>/
  best_model.zip          final_model.zip
  evaluations.npz         monitor/*.monitor.csv
  tensorboard/            videos/
  checkpoints/            eval_info.csv
  vec_normalize.pkl       best_vec_normalize.pkl
  config.json             stage_summary.txt
  learning_curve.png      eval_info.png
  training_health.png     best_model.mp4
```

`checkpoints/` holds periodic full-state snapshots from `CheckpointCallback`; `eval_info.csv` is the long-format mirror of `InfoDictEvalCallback`'s TensorBoard scalars (timestep, metric, value). Two `VecNormalize` snapshots are written: `best_vec_normalize.pkl` captures the obs-normalization stats at the moment `best_model.zip` was saved, while `vec_normalize.pkl` holds the end-of-training stats. `record_best_model_video` prefers the former, so replay normalizes observations exactly the way the best model saw them.

Root is `MyDrive/Finding Theta/courtside-dynamics/training_runs/` when Drive is mounted, otherwise `./logs/`.


In [ ]:
from courtside_dynamics.notebook_utils import mount_drive, resolve_run_dir
from courtside_dynamics.recipes import RECIPES

if USE_DRIVE:
    mount_drive()

ALGO = ALGO or RECIPES[ENV].default_algo
LOG_DIR = resolve_run_dir(ENV, ALGO, use_drive=USE_DRIVE)
print("Logging to:", LOG_DIR)

## 4. Configure Colab GPU

Sets up EGL so MuJoCo can render off-screen on the Colab GPU. No-op outside of Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab
setup_colab()

## 5. Build the training config

`build_train_config` looks up the recipe for `ENV`, fills in the per-env extras (e.g. custom CSV rows for Ball Bounce), and returns a `TrainConfig` ready for `train()`.

In [ ]:
from courtside_dynamics.recipes import RECIPES, build_train_config

print(f"Recipe: {ENV} -> {RECIPES[ENV].description}")

# Resolve the worker count: explicit override, then recipe-specific
# default, then the generic per-algo suggestion from section 2.
recipe_n_envs = RECIPES[ENV].extra_cfg.get("n_envs")
n_envs = N_ENVS if N_ENVS is not None else (recipe_n_envs if recipe_n_envs is not None else (8 if ALGO.upper() == "SAC" else 16))

cfg = build_train_config(
    ENV,
    algo=ALGO,
    log_dir=LOG_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    quick_test=QUICK_TEST,
    seed=SEED,
    n_envs=n_envs,
    early_stop_patience=EARLY_STOP_PATIENCE,
    model_kwargs=MODEL_KWARGS,
)

probe_env = cfg.env_fn()
try:
    print(f"spaces: action={probe_env.action_space.shape} observation={probe_env.observation_space.shape}")
finally:
    probe_env.close()

print(
    f"algo={cfg.algo}  total_timesteps={cfg.total_timesteps:,}  "
    f"n_envs={cfg.n_envs}  seed={cfg.seed}  eval_freq={cfg.eval_freq:,}  "
    f"early_stop_patience={cfg.early_stop_patience}  "
    f"checkpoint_freq={cfg.checkpoint_freq:,}  "
    f"video_freq={cfg.video_freq:,}  model_kwargs={cfg.model_kwargs}"
)


## 5b. Live TensorBoard (optional)

Starts an inline TensorBoard tailing `LOG_DIR/tensorboard` so a multi-hour run can be checked mid-flight: eval reward under `eval/`, optimizer health under `train/`, per-eval info metrics under `eval_info/`. Scalars appear after SB3's first metric dump; use the refresh button. Safe to skip -- every scalar shown here is also mirrored to `progress.csv` / `eval_info.csv` and plotted statically in sections 7-8b.

In [ ]:
import os

from tensorboard import notebook as tb_notebook

# notebook.start shlex-parses its args, so the quotes keep Drive
# paths with spaces (MyDrive/Finding Theta/...) intact.
tb_notebook.start(f'--logdir "{os.path.join(LOG_DIR, "tensorboard")}"')


## 6. Train

`train(cfg)` builds vectorized train + eval envs, attaches `EvalCallback`, `VideoRecordCallback`, and `InfoDictEvalCallback`, and runs SB3's `model.learn`. The best policy seen during evaluation is saved to `LOG_DIR/best_model.zip` (with its matching `best_vec_normalize.pkl`), so a later collapse never loses it. With `EARLY_STOP_PATIENCE` set, training stops automatically once evaluations plateau -- the stop reason prints below and `stage_summary.txt` records how many steps of the budget were actually used.

In [ ]:
from courtside_dynamics.training import train

model = train(cfg)

## 6b. Run report

`train()` writes `stage_summary.txt` at the end of every run -- final/best eval, wall-clock duration, throughput, device, and the final `train/*` health metrics -- including interrupted runs (`status: interrupted`). Printing it here attaches the numbers to this notebook session, and it's the first thing to paste when asking "why did this run underperform?".

In [ ]:
from courtside_dynamics.notebook_utils import print_stage_summary

print_stage_summary(LOG_DIR)


## 7. Learning curves

Per-episode training rewards (left) come from `LOG_DIR/monitor/*.monitor.csv`. Deterministic eval rewards (right, mean +/- std) come from `LOG_DIR/evaluations.npz`.

In [ ]:
import os
from courtside_dynamics.notebook_utils import plot_learning_curve

plot_learning_curve(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "learning_curve.png"),
)

## 8. Eval-info curves

One panel per scalar `info` key tracked by `InfoDictEvalCallback` (rally count, paddle / wall touches, phase fractions, ...). `_mean`, `_final`, and `_max` variants are overlaid as separate lines per panel. The data comes from `LOG_DIR/eval_info.csv` (long-format mirror of the TensorBoard scalars, written every eval).

In [ ]:
from courtside_dynamics.notebook_utils import plot_eval_info

plot_eval_info(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "eval_info.png"),
)

## 8b. Training-health curves

SB3's own optimizer diagnostics from `LOG_DIR/tensorboard/progress.csv`. For **SAC**: `ent_coef` (the entropy temperature — watch for it collapsing too fast or sticking high), plus `actor_loss` / `critic_loss` (a diverging critic is the classic failure). For **PPO**: `explained_variance` (below 0 means the value function is worse than predicting the mean), `approx_kl`, `clip_fraction`. These explain a stalled run that the reward curve alone won't.

In [ ]:
from courtside_dynamics.notebook_utils import plot_training_health

plot_training_health(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "training_health.png"),
)

## 9. Replay the best model

Loads `best_model.zip` from `LOG_DIR`, rolls it out deterministically, encodes the frames as MP4, and embeds the clip in this notebook.

In [ ]:
from courtside_dynamics.notebook_utils import (
    record_best_model_video,
    display_video,
)

video_path = record_best_model_video(
    LOG_DIR,
    cfg.env_fn,
    algo=ALGO,
    video_length=750,
)
display_video(video_path)

## 9b. Artifact audit

Checks `LOG_DIR` against every artifact this notebook should have produced and prints the most likely cause for anything missing (video skipped because moviepy failed, no `best_model.zip` because eval never fired, ...). Run it **before** disconnecting: it's the last chance to re-run a failed cell while the runtime -- and everything not synced to Drive -- still exists.

In [ ]:
from courtside_dynamics.notebook_utils import check_run_artifacts

missing = check_run_artifacts(LOG_DIR)


## 10. Disconnect Colab runtime

Frees the GPU once the whole pipeline -- training, plots, replay video, artifact audit -- has finished, so an unattended Run-All doesn't hold the runtime for hours after the work is done. Everything above is already saved under `LOG_DIR` (on Drive when `USE_DRIVE=True`) and audited in section 9b. Comment the cell out if you want to keep the session alive for interactive inspection. No-op outside Colab.

In [ ]:
# Frees the GPU when the run is over. All artifacts are already on
# disk (and Drive when USE_DRIVE=True) and audited above; comment
# this out to keep the runtime for interactive inspection. No-op
# outside Colab. The delay lets the final cell outputs render.
from courtside_dynamics.notebook_utils import disconnect_runtime

disconnect_runtime(delay_seconds=30)
